# XGBoost 모델 구성 설명

## 목표와 Target

- 목표: 학생별로 **해당 예측 주차 종료 시점에서 다음 주 이탈확률**을 예측합니다.
- 사용 데이터: `demo_1/used_data/weekly_next_week_with_vle_enhanced.csv`
- Target: `target_next_week_withdrawn` — 다음 주 이탈이면 1, 아니면 0

## 사용 Feature

모델 입력은 `id_student`와 Target을 제외한 **124개 Feature**입니다. `id_student`는 모델 입력에 넣지 않고 GroupKFold 분할 기준으로만 사용합니다.

- 과목·운영·주차: `code_module`, `code_presentation`, `prediction_week`, `cutoff_day`, 강좌 운영 길이
- 학생 특성: 성별, 지역, 최종 학력, IMD 구간, 연령대, 이전 수강 횟수, 수강 학점, 장애 여부
- 등록 Feature: 등록일, 개강 후 등록 여부, 사전 등록일 수, 지연 등록일 수
- 평가 누적 Feature: 평가 수·비중·제출 수·지각 제출 수·미제출 수·제출률·평균 점수
- VLE 누적 Feature: 누적 클릭 수·활동일 수·고유 사이트 수·마지막 활동일·활동 유형별 누적 클릭 수·VLE 기록 여부

모든 평가·VLE Feature는 `prediction_week` 종료일까지 누적된 값만 사용하며, 미래 정보와 이탈 후 행은 제외되어 있습니다.

## 전처리·검증

- 범주형 8개(`code_module`, `code_presentation`, 성별·지역·학력·IMD·연령·장애 여부)는 `OneHotEncoder(handle_unknown='ignore')`로 변환합니다.
- 수치형 Feature는 그대로 사용하고 무한값은 결측값으로 바꿉니다.
- `id_student` 기준 **3-Fold GroupKFold**로 분할해 동일 학생의 여러 주차 행이 Train과 Test에 동시에 들어가지 않게 합니다.
- One-Hot Encoder는 각 Train Fold에만 학습해 Test Fold의 범주 정보가 사전에 유입되지 않게 합니다.

## 하이퍼파라미터

| 항목 | 설정값 | 의미 |
|---|---:|---|
| `objective` | `binary:logistic` | 이진 이탈확률 예측 |
| `eval_metric` | `aucpr` | 불균형 데이터용 PR-AUC 기준 |
| `n_estimators` | 500 | 최대 트리 수 |
| `learning_rate` | 0.05 | 작은 학습률로 점진적 학습 |
| `max_depth` | 6 | 트리 최대 깊이 |
| `min_child_weight` | 5 | 과도한 세분화를 제한 |
| `subsample`, `colsample_bytree` | 0.8, 0.8 | 행·Feature 일부 샘플링으로 과적합 완화 |
| `reg_lambda` | 5.0 | L2 규제 강도 |
| `scale_pos_weight` | Fold별 비이탈자 수 / 이탈자 수 | 낮은 이탈 비율 보정 |
| `tree_method` | `hist` | 대용량 데이터의 빠른 히스토그램 학습 |
| `early_stopping_rounds` | 40 | 검증 PR-AUC가 개선되지 않으면 조기 종료 |
| `random_state` | 42 | 재현성 확보 |

## 학습·예측·평가 흐름

각 Fold에서 Train 데이터로 전처리와 모델 학습을 수행한 뒤, Test Fold의 다음 주 이탈확률을 예측합니다. 세 Fold의 Test 예측을 합친 OOF 확률로 `Recall@Top-20%`, `PR-AUC`, `Brier Score`, `ECE(10구간)`를 계산합니다.

# 확장 데이터 재학습 결과

- 사용 데이터: `demo_1/used_data/weekly_next_week_with_vle_enhanced.csv`
- 데이터 크기: 895,005행 × 126열, 모델 입력 Feature 124개
- 검증 방식: `GroupKFold(id_student, 3-Fold)`
- OOF Recall@Top-20%: **0.720773**
- OOF PR-AUC: **0.085647**
- Brier Score: **0.137278**
- ECE(10-bin): **0.269333**

> 이 수치는 확장 데이터로 재학습한 결과이다. 기존 53열 데이터 성능과 비교할 때는 동일한 GroupKFold 조건을 유지한다.


# XGBoost: 주차별 다음 주 이탈 예측

`weekly_next_week_with_vle_enhanced.csv`를 사용합니다. Target은 `target_next_week_withdrawn`이며, 동일 학생의 여러 주차 행이 학습·검증에 함께 섞이지 않도록 `id_student` 기준 GroupKFold를 적용합니다.

In [ ]:
import runpy
runpy.run_path('01_xgboost_weekly_next_week.py', run_name='__main__')

In [ ]:
import pandas as pd
pd.read_csv('xgboost_weekly_next_week_metrics.csv')